# Educational LLaDA-style Diffusion Language Model

A compact TensorFlow/Keras implementation of a **masked discrete-diffusion language model inspired by LLaDA**.

This notebook is intentionally described as **LLaDA-style**, not as a drop-in reproduction of the reference implementation. It keeps the central ideas—bidirectional attention, a dedicated absorbing `MASK` state, masked-token training, iterative denoising, confidence-based remasking, and optional unsupervised classifier-free guidance (CFG)—while making several educational/experimental choices explicit.

### What this notebook implements

- Fully bidirectional Transformer (`use_causal_mask=False`).
- Dedicated `MASK` token distinct from EOS.
- Per-sequence masking level and masked-token-only cross-entropy.
- Random and low-confidence reverse-diffusion remasking, plus optional semi-autoregressive block variants.
- Optional unsupervised CFG using a prompt-masked unconditional branch.
- Exponential moving average (EMA) weights for evaluation/sampling.
- Memory-saving sparse vocabulary projection at masked positions.
- Fixed-corruption metrics plus pseudo-perplexity (PPP) diagnostics.

### Deliberate educational / experimental choices

These are **not required parts of the basic LLaDA objective**:

- An easy-to-hard corruption curriculum that eventually reaches `t_max = 1.0`.
- An optional early-training repetition regularizer (`rep_schedule_peak`; set it to `0.0` to disable).
- Optional Gumbel-max temperature annealing during generation (`tau_start` / `tau_end`; set both to `0.0` for greedy sampling).
- An optional left-token sampling repetition penalty (`repeat_penalty`; set to `0.0` to disable).
- A short guidance-off bootstrap at the beginning of CFG generation; the prompt-masked CFG formula itself follows the unsupervised CFG form described for LLaDA.

### Presets

- `quick`: smaller model, capped training text, capped validation workload; intended for end-to-end verification and teaching.
- `full`: larger 1024-dimension / 12-layer / 16-head configuration and the full validation set.

> `ppl@t=0.15` and PPP are **not directly comparable to autoregressive perplexity** because the bidirectional model sees context on both sides of a masked token.

### Notebook workflow

Part I defines the building blocks; Part II runs each expensive stage in its own cell: data loading, tokenization, window construction, training, generation, masked-BPT evaluation, and PPP evaluation. Each stage prints its own timing, so it is easy to see what is running and where a slow step is.

Paper: *Large Language Diffusion Models* (LLaDA), Nie et al., 2025.


# Part I — Building blocks

Sections 1–8 only **define** functions and classes; nothing is downloaded or trained yet. Part II (from section 9) runs the pipeline stage by stage.

## 1. Setup, configuration, model, and forward corruption

Let $x_0$ be a clean token sequence. For each sequence we sample a corruption level $t$, then independently replace tokens with the dedicated `MASK` state with probability $t$.

The training loss is cross-entropy **only at masked positions**, normalized per sequence by its realized number of masked tokens and then averaged across the batch.

Two details in this notebook are deliberate deviations from a minimal/reference-style objective:

1. The corruption distribution follows an easy-to-hard curriculum.
2. An optional repetition regularizer is used early in training. Set `rep_schedule_peak=0.0` (or the component weights to zero) to disable it.

The Transformer is fully bidirectional: `use_causal_mask=False`.


In [ ]:
import os
import time
os.environ.pop("TF_GPU_ALLOCATOR", None)  # avoid cuda_malloc_async fragmentation
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # must be set before importing TensorFlow

import json, math
import re, html, unicodedata
from dataclasses import dataclass, asdict

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
import matplotlib.pyplot as plt
from IPython.display import clear_output
import tiktoken
import absl.logging

absl.logging.set_verbosity(absl.logging.ERROR)

gpus = tf.config.list_physical_devices("GPU")
mixed_precision.set_global_policy("mixed_float16" if gpus else "float32")

try:
    if gpus:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
except Exception as _e:
    pass
# Avoid large XLA fusion buffers on proj steps; not needed here.
tf.config.optimizer.set_jit(False)

# Safe scalar converters for eager Tensors
def _to_python_int(x):
    return int(x) if isinstance(x, (int, np.integer)) else int(x.numpy())
def _to_python_float(x):
    return float(x) if isinstance(x, (float, np.floating)) else float(x.numpy())

@dataclass
class LLaDAConfig:
    vocab_size: int = 50_258
    context_length: int = 128
    emb_dim: int = 384
    n_heads: int = 6
    num_transformer_layers: int = 6
    drop_rate: float = 0.1
    num_sampling_steps: int = 256  # number of reverse-diffusion sampling steps
    mask_token_id: int = 50_257  # dedicate a MASK/absorbing state (separate from text)
    eos_token_id: int = 50_256   # GPT-2 EOS; keep distinct from MASK
    guidance_strength: float = 0.6  # unsupervised CFG (prompt-masked unconditional branch, as in LLaDA); set 0.0 to disable
    block_size: int = 32  # for semi-autoregressive block sampling
    sampling_tau_start: float = 1.5  # non-LLaDA heuristic; set both tau values to 0 for greedy sampling
    sampling_tau_end: float = 0.3
    sampling_repeat_penalty: float = 4.0  # non-LLaDA heuristic; set 0 to disable
    # --- repetition loss knobs (training-time) ---
    rep_w_immediate: float = 0.25   # penalize P(y == left token)
    rep_w_two_back: float  = 0.10   # penalize P(y == token two steps back)
    rep_w_presence: float  = 0.05   # small presence penalty on recent tokens (set 0.0 to disable)
    rep_presence_window: int = 32   # lookback window in tokens for presence set
    rep_presence_k: int = 8         # sample last-K tokens from the window (0 disables presence part)
    rep_schedule_peak: float = 0.1  # non-LLaDA auxiliary-loss scale; set 0.0 to disable

# Unsupervised CFG (LLaDA): mix conditional and prompt-masked branches; not related to remasking.
# The guidance-off bootstrap for the first few steps in `llada_generate` is this notebook's own addition.
def _cfg_combine(cond_logits, uncond_logits, guidance_strength):
    """Classifier-free guidance with 0.0 as the neutral setting."""
    if guidance_strength is None or guidance_strength <= 0.0:
        return cond_logits
    return cond_logits + guidance_strength * (cond_logits - uncond_logits)

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, vocab_size, context_length, emb_dim, drop_rate=0.1):
        super().__init__()
        init = keras.initializers.RandomNormal(stddev=0.02)
        self.tok_emb = layers.Embedding(vocab_size, emb_dim, embeddings_initializer=init)  # keep vars in fp32 under mixed precision
        self.pos_emb = layers.Embedding(context_length, emb_dim, embeddings_initializer=init)  # fp32 vars; AMP will cast outputs
        self.drop = layers.Dropout(drop_rate)
        self.context_length = context_length

    def call(self, x, training=False):
        # x: (B, T)
        T = tf.shape(x)[1]
        positions = tf.range(start=0, limit=T, delta=1)
        tok = self.tok_emb(x)                    # (B, T, C)
        pos = self.pos_emb(positions)[None, ...] # (1, T, C)
        # Both embeddings use GPT-2-style stddev=0.02 initialization. Scaling only
        # token embeddings by sqrt(d_model) would weaken the positional signal.
        out = tok + pos
        return self.drop(out, training=training)

class TransformerBlock(layers.Layer):
    def __init__(self, emb_dim, n_heads, drop_rate=0.1):
        super().__init__()
        assert emb_dim % n_heads == 0, "emb_dim must be divisible by n_heads"
        self.ln1 = layers.LayerNormalization(epsilon=1e-5, dtype="float32")
        self.attn = layers.MultiHeadAttention(
            num_heads=n_heads,
            key_dim=emb_dim // n_heads,
            dropout=drop_rate,
            kernel_initializer=keras.initializers.RandomNormal(stddev=0.02),
        )
        self.ln2 = layers.LayerNormalization(epsilon=1e-5, dtype="float32")
        self.ffn = keras.Sequential([
            layers.Dense(4*emb_dim, activation=keras.activations.gelu,
                         kernel_initializer=keras.initializers.RandomNormal(stddev=0.02)),
            layers.Dense(emb_dim, kernel_initializer=keras.initializers.RandomNormal(stddev=0.02)),
            layers.Dropout(drop_rate),
        ])

        self.resid_drop1 = layers.Dropout(drop_rate)
        self.resid_drop2 = layers.Dropout(drop_rate)

    def call(self, x, training=False):
        h = self.ln1(x, training=training)
        attn_out = self.attn(query=h, value=h, key=h, use_causal_mask=False, training=training)
        x = x + self.resid_drop1(attn_out, training=training)
        h2 = self.ln2(x, training=training)
        x = x + self.resid_drop2(self.ffn(h2, training=training), training=training)
        return x

class LLaDAModel(keras.Model):
    def __init__(self, config: LLaDAConfig):
        super().__init__()
        self.config = config
        self.embed = TokenAndPositionEmbedding(
            config.vocab_size, config.context_length, config.emb_dim, config.drop_rate
        )
        self.transformer_blocks = [
            TransformerBlock(config.emb_dim, config.n_heads, config.drop_rate)
            for _ in range(config.num_transformer_layers)
        ]
        self.ln_f = layers.LayerNormalization(epsilon=1e-5, dtype="float32")
        self.head_bias = self.add_weight(shape=(config.vocab_size,), initializer="zeros", name="head_bias")
        self._mask_bias = None  # set in build()

    def build(self, input_shape):
        # Precompute a constant vector with -1e4 at MASK index, 0 elsewhere.
        vb = self.config.vocab_size
        mi = self.config.mask_token_id
        # Create once; no per-call one_hot. Keep non-trainable.
        vec = tf.tensor_scatter_nd_update(
            tf.zeros([vb], dtype=self.head_bias.dtype),
            indices=[[mi]],
            updates=[tf.constant(-1e4, dtype=self.head_bias.dtype)],
        )
        # Store as a constant; cast on use to match logits dtype.
        self._mask_bias = tf.constant(vec, dtype=self.head_bias.dtype)
        super().build(input_shape)

    def _forbid_mask(self, logits):
        # Broadcast adds [V] -> [B,T,V]; cast for mixed precision safety.
        return logits + tf.cast(self._mask_bias, logits.dtype)

    def encode(self, x, training=False):
        # Return hidden states before vocab projection
        x = x[:, -self.config.context_length:]
        h = self.embed(x, training=training)
        for blk in self.transformer_blocks:
            h = blk(h, training=training)
        return tf.cast(self.ln_f(h, training=training), tf.keras.mixed_precision.global_policy().compute_dtype)

    def logits_at_positions(self, h, pos_idx):
        """
        h: [B, T, C], pos_idx: [B] int32 positions per row.
        Returns logits ONLY at those positions: [B, V]
        """
        B = tf.shape(h)[0]
        idx = tf.stack([tf.range(B, dtype=tf.int32), tf.cast(pos_idx, tf.int32)], axis=1)
        h_sel = tf.gather_nd(h, idx)                               # [B, C]
        E  = self.embed.tok_emb.embeddings                                # [V, C]
        hb = self.head_bias
        logits_half = tf.matmul(h_sel, tf.cast(E, h_sel.dtype), transpose_b=True) + tf.cast(hb, h_sel.dtype)
        return self._forbid_mask(logits_half)

    def call(self, x, training=False):
        # x: [B,T] with some tokens possibly MASK
        x = x[:, -self.config.context_length:]
        h = self.embed(x, training=training)
        for blk in self.transformer_blocks:
            h = blk(h, training=training)
        h = tf.cast(self.ln_f(h, training=training), tf.keras.mixed_precision.global_policy().compute_dtype)
        B = tf.shape(h)[0]; T = tf.shape(h)[1]; C = tf.shape(h)[2]

        # project in mixed precision (no fp32 up-cast)
        h_flat = tf.reshape(h, [-1, C])                                    # [B*T, C] in compute dtype (fp16/bf16)
        E      = self.embed.tok_emb.embeddings                             # [V, C] (stored fp32, cast on-the-fly)
        hb     = self.head_bias
        logits_half = tf.matmul(h_flat, tf.cast(E, h_flat.dtype), transpose_b=True) + tf.cast(hb, h_flat.dtype)
        logits      = tf.reshape(tf.cast(logits_half, h.dtype), [B, T, tf.shape(E)[0]])
        return self._forbid_mask(logits)
    
class WarmupStableDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, hold_steps, decay_steps, min_lr=1e-6):
        self.base_lr = tf.convert_to_tensor(base_lr, tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.hold_steps = tf.cast(hold_steps, tf.float32)
        self.decay_steps = tf.cast(tf.maximum(decay_steps, 1), tf.float32)
        self.min_lr = tf.convert_to_tensor(min_lr, tf.float32)
    def __call__(self, step):
        s = tf.cast(step, tf.float32)
        # warmup
        lr_w = self.base_lr * tf.minimum(s / tf.maximum(self.warmup_steps, 1.0), 1.0)
        # hold
        after_w = tf.maximum(s - self.warmup_steps, 0.0)
        # linear decay
        after_wh = tf.maximum(after_w - self.hold_steps, 0.0)
        frac = tf.minimum(after_wh / self.decay_steps, 1.0)
        lr_d = self.base_lr - (self.base_lr - self.min_lr) * frac
        return tf.where(s < self.warmup_steps, lr_w, tf.where(after_w < self.hold_steps, self.base_lr, lr_d))

def sample_t_stratified(batch_size, t_min=0.10, t_max=1.0):
    """Sample one jittered stratum per batch element, then shuffle across the batch."""
    B = tf.cast(batch_size, tf.int32)
    u = (tf.cast(tf.range(B), tf.float32) + tf.random.uniform([B], dtype=tf.float32)) / tf.cast(B, tf.float32)
    u = tf.random.shuffle(u)
    return tf.cast(t_min, tf.float32) + (tf.cast(t_max, tf.float32) - tf.cast(t_min, tf.float32)) * u

def mask_with_given_t(x0, mask_token_id, t_vec):
    """Mask with per-example t values (shape [B])."""
    u = tf.random.uniform(tf.shape(x0), dtype=tf.float32)
    mask_bool = u < t_vec[:, None]
    xt = tf.where(mask_bool, tf.fill(tf.shape(x0), tf.cast(mask_token_id, x0.dtype)), x0)
    return xt, mask_bool, t_vec

def mask_exact_k_given_t(x0, mask_token_id, t_vec):
    """Mask exactly k = round(t*L) tokens per example (uniform without replacement)."""
    B, L = tf.shape(x0)[0], tf.shape(x0)[1]
    noise = tf.random.uniform([B, L])
    ranks = tf.argsort(noise, axis=-1)  # small rank = earlier
    k = tf.cast(tf.round(t_vec * tf.cast(L, tf.float32)), tf.int32)
    k = tf.clip_by_value(k, 1, L)
    mask_bool = tf.cast(ranks, tf.int32) < k[:, None]
    xt = tf.where(mask_bool, tf.fill(tf.shape(x0), tf.cast(mask_token_id, x0.dtype)), x0)
    t_hat = tf.cast(k, tf.float32) / tf.cast(L, tf.float32)
    return xt, mask_bool, t_hat

#=========================================================================================================================================



## 2. WikiText loading, cleaning, tokenization, and windows

The loader caches the **full raw split** and applies `subset_ratio` / `max_chars` only at runtime. This avoids a subtle reproducibility problem where a quick-start run could permanently overwrite the local cache with a truncated corpus.

WikiText escape artifacts such as `@-@` and `@,@` are normalized before tokenization.


In [ ]:
# ----------------------- Data -----------------------
def load_wikitext_cached(
    dataset_id: str = "wikitext-103-v1",
    local_dir: str = "data/wikitext103",
    force_download: bool = False,
    subset_ratio: float | None = None,
    max_chars: int | None = None,
) -> tuple[str, str]:
    """Load WikiText, cache the full raw splits, then apply optional runtime limits."""
    from datasets import load_dataset, DownloadConfig

    os.makedirs(local_dir, exist_ok=True)
    train_path = os.path.join(local_dir, f"{dataset_id}_train.txt")
    valid_path = os.path.join(local_dir, f"{dataset_id}_valid.txt")

    if (not force_download) and os.path.exists(train_path) and os.path.exists(valid_path):
        with open(train_path, "r", encoding="utf-8") as f:
            train_text = f.read()
        with open(valid_path, "r", encoding="utf-8") as f:
            valid_text = f.read()
    else:
        hf_cache_dir = os.path.join(local_dir, "hf_cache")
        os.makedirs(hf_cache_dir, exist_ok=True)
        try:
            ds = load_dataset(
                "wikitext",
                dataset_id,
                cache_dir=hf_cache_dir,
                download_config=DownloadConfig(local_files_only=True),
                download_mode="reuse_dataset_if_exists",
            )
        except Exception as offline_err:
            try:
                dm = "force_redownload" if force_download else "reuse_dataset_if_exists"
                ds = load_dataset("wikitext", dataset_id, cache_dir=hf_cache_dir, download_mode=dm)
            except Exception as online_err:
                raise RuntimeError(
                    f"Could not load WikiText dataset '{dataset_id}'. Tried offline, then online.\n"
                    f"Offline error: {offline_err}\nOnline error: {online_err}"
                ) from online_err

        def _concat_nonempty(split: str) -> str:
            return "\n\n".join(t for t in ds[split]["text"] if t and t.strip())

        train_text = _concat_nonempty("train")
        valid_text = _concat_nonempty("validation")
        with open(train_path, "w", encoding="utf-8") as f:
            f.write(train_text)
        with open(valid_path, "w", encoding="utf-8") as f:
            f.write(valid_text)

    def _prepare(text: str) -> str:
        # Apply the same quick-start limits whether the corpus came from cache or a fresh download.
        if subset_ratio is not None and 0.0 < subset_ratio < 1.0:
            lines = [line for line in text.splitlines() if line.strip()]
            text = "\n\n".join(lines[:max(1, int(len(lines) * subset_ratio))])
        if max_chars is not None and max_chars > 0:
            text = text[:max_chars]
        return clean_wikitext_raw(text) if "raw" in dataset_id else text

    return _prepare(train_text), _prepare(valid_text)


def clean_wikitext_raw(text: str) -> str:
    # Preserve top-level WikiText article boundaries before removing headings.
    text = re.sub(r"(?m)^\s*=\s+[^=\n]+\s+=\s*$", "\n@@ARTICLE_BOUNDARY@@\n", text)

    # Drop HTML-ish tags and references.
    text = re.sub(r"<ref[^>]*>.*?</ref>", "", text, flags=re.DOTALL|re.IGNORECASE)
    text = re.sub(r"<[^>]+>", "", text)

    # Remove simple (non-nested) templates {{...}}.
    text = re.sub(r"\{\{[^{}]*\}\}", "", text)

    # Resolve wiki links [[target|alias]] -> alias (or target if no alias).
    def _link_sub(m):
        return m.group(2) if m.group(2) else m.group(1)

    text = re.sub(r"\[\[([^|\]]+)\|?([^\]]*)\]\]", _link_sub, text)

    # Drop remaining subsection headings and file/category lines.
    text = re.sub(r"^\s*=+.*?=+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*(File|Image|Category):.*$", "", text, flags=re.MULTILINE|re.IGNORECASE)
    # Undo WikiText escaped in-word punctuation: "cup @-@ like" -> "cup-like", "1 @,@ 000" -> "1,000".
    text = re.sub(r" @([-.,])@ ", r"\1", text)

    # Normalize whitespace while preserving article separators.
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def prettify_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", html.unescape(s))
    # tighten spaces around punctuation
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\(\s+", "(", s); s = re.sub(r"\s+\)", ")", s)
    s = re.sub(r"\s{2,}", " ", s)
    return s.strip()

def get_tokenizer():
    return tiktoken.get_encoding("gpt2")  # GPT-2 BPE

def encode_in_chunks(text, tokenizer, chunk_chars=1_000_000):
    """Encode text and insert GPT-2 EOS at preserved WikiText article boundaries."""
    ids = []
    eos = tokenizer.eot_token

    for article in text.split("@@ARTICLE_BOUNDARY@@"):
        article = article.strip()
        if not article:
            continue
        for i in range(0, len(article), chunk_chars):
            ids.extend(tokenizer.encode(article[i:i + chunk_chars]))
        ids.append(eos)

    return ids

# ----------------------- Windows ------------------------
def build_windows(token_ids, context_length: int, stride: int):
    if len(token_ids) < context_length:
        return np.empty((0, context_length), dtype=np.int32)
    inputs, i = [], 0
    L = len(token_ids) - (context_length)
    while i <= L:
        x = token_ids[i : i + context_length]
        inputs.append(x)
        i += stride
    X = np.array(inputs, dtype=np.int32)
    return X  # targets are the clean x0 itself

def make_dataset(X, batch_size=16, shuffle=True, drop_remainder=True):
    ds = tf.data.Dataset.from_tensor_slices(X)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 10000), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds



## 3. Training visualization, EMA, and carry-over logits

`EMA` means **exponential moving average** of model weights. Sampling and validation are performed with EMA weights to reduce noise.

The plotted curriculum-weighted validation loss is *not* used for checkpoint selection because its corruption distribution changes over training. Checkpoints are selected using fixed corruption CE at `t=0.15`.


In [ ]:
# ----------------------- Plot panel --------
def plot_lm_training_one_panel(
    train_losses, val_losses,
    train_ppls,   val_ppls,
    lrs, best_round, logy=False
):
    c_train_loss = "tab:orange";  c_val_loss = "tab:orange"
    c_train_ppl  = "tab:green";   c_val_ppl  = "tab:green"
    c_lr         = "tab:gray";    c_best     = "tab:red"

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.plot(train_losses, lw=2.0, color=c_train_loss, label="Train • Loss")
    ax.plot(val_losses,   lw=5.0, ls="--", color=c_val_loss, label="Val • Loss")
    if logy: ax.set_yscale("log")
    ax.set_xlabel("Training round"); ax.set_ylabel("Loss"); ax.grid(alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(train_ppls, lw=2.0, color=c_train_ppl, label="Train • PPL @ t=0.15")
    ax2.plot(val_ppls,   lw=5.0, ls="--", color=c_val_ppl, label="Val • PPL @ t=0.15")
    # draw dots:
    # 1) Best Val PPL: choose round_idx where val_ppls is minimal (ignoring NaNs)
    val_ppls_np = np.asarray(val_ppls, dtype=float)
    if np.any(np.isfinite(val_ppls_np)):
        be_ppl = int(np.nanargmin(val_ppls_np))
        ppl_value = float(val_ppls_np[be_ppl])
        ax2.scatter([be_ppl], [ppl_value], s=90, zorder=3, color=c_best, edgecolor="white", linewidth=0.9, label="Best Val PPL")
        ax2.annotate(f"{ppl_value:.1f}", xy=(be_ppl, ppl_value), xytext=(0, 8), textcoords="offset points", ha="center", va="bottom",
                     fontsize=10, color=c_best, weight="bold", bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))

    # 2) Saved checkpoint marker. best_round is selected by fixed CE @ t=0.15,
    # not by the moving-curriculum validation loss shown on the left axis.
    if best_round is not None:
        be_loss = int(best_round)
        if 0 <= be_loss < len(val_losses) and np.isfinite(val_losses[be_loss]):
            ax.scatter([be_loss], [val_losses[be_loss]], s=90, marker="X", zorder=3, color=c_val_loss,
                       label="Saved checkpoint (best val CE@t=0.15)")

    ax2.set_ylabel("Masked perplexity (fixed t=0.15)")
    ax2.set_yscale("log")
    ax3 = ax.twinx()
    ax3.spines.right.set_position(("axes", 1.08))
    ax3.plot(lrs, lw=1.9, alpha=0.95, color=c_lr, label="Learning rate")
    ax3.set_ylabel("Learning rate"); ax3.set_yscale("log")

    ax.set_title("Discrete diffusion LM training: loss, fixed-corruption masked perplexity, learning rate")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()
    ax.legend(lines1 + lines2 + lines3, labels1 + labels2 + labels3, ncol=3, loc="upper left")
    plt.tight_layout(); plt.show()


class EMA:
    def __init__(self, model, decay=0.999):
        # Track trainable vars only (matches optimizer updates; lighter & safer)
        self.shadow = [tf.Variable(tf.identity(v), trainable=False) for v in model.trainable_weights]
        self.decay = decay
        self.model = model
    def update(self):
        for s, v in zip(self.shadow, self.model.trainable_weights):
            s.assign(self.decay * s + (1.0 - self.decay) * v)
    def copy_to(self, dst_model):
        for s, v in zip(self.shadow, dst_model.trainable_weights):
            v.assign(s)

# Helper
def apply_carry_over_logits(logits, xk, mask_id):
    """
    logits: [B,T,V] float
    xk:     [B,T]   int32 (noised inputs at time t)
    Sets logits at unmasked positions (xk != mask_id) to be a delta at xk.
    """
    V = tf.shape(logits)[-1]
    # one-hot over the token we want to copy through
    copy_onehot = tf.one_hot(xk, V, dtype=logits.dtype)                      # [B,T,V]
    # keep finite in fp16 to avoid -inf propagation under XLA
    neg_inf = tf.cast(tf.constant(-1e4, dtype=tf.float32), logits.dtype)
    copied = (1.0 - copy_onehot) * neg_inf
    # only apply where NOT masked
    unmasked = tf.not_equal(xk, tf.cast(mask_id, xk.dtype))                  # [B,T]
    unmasked = tf.cast(unmasked, logits.dtype)[..., None]
    return unmasked * copied + (1.0 - unmasked) * logits



## 4. Reverse diffusion sampler

A reverse step has the following structure:

**masked state $x_t$** → **bidirectional Transformer** → **token logits** → **fill masked positions** → **confidence ranking** → **remask selected positions** → **state $x_s$**

where $t=\text{step}/K$, $s=(\text{step}-1)/K$, and $K$ is the number of reverse-diffusion steps.

### Sampling temperature

The Gumbel-max identity is important:

$$
\arg\max_i \left(z_i + \tau G_i\right)
$$

samples from `softmax(z / τ)` when $G_i$ are i.i.d. Gumbel random variables. Therefore `tau` is a **sampling temperature**, not merely a tie-break magnitude.

- `tau = 0`: pure greedy sampling.
- `tau > 0`: stochastic sampling.
- The enhanced default in this notebook anneals from `1.5` early to `0.3` late.

### CFG and notebook-specific sampling heuristics

The optional CFG branch masks the prompt to construct an unconditional prediction and combines conditional/unconditional logits with the usual unsupervised-CFG form.

This notebook additionally uses two optional sampling heuristics: Gumbel-max temperature annealing and a left-token repetition penalty. It also keeps CFG off for the first few all/mostly-masked reverse steps before enabling the configured guidance strength.

For a conservative deterministic setting with these extras disabled, call `llada_generate(..., guidance_strength=0.0, tau_start=0.0, tau_end=0.0, repeat_penalty=0.0)`.


In [ ]:
# ----------------------- Masking helpers ----------------
def llada_reverse_step(model, x_t, step, num_steps, mask_token_id, prompt_len=0, strategy="low_confidence", guidance_strength=0.0, block_size=16, tau_start=1.5, tau_end=0.3, repeat_penalty=4.0):
    """
    One reverse step t=step/num_steps -> s=(step-1)/num_steps. Sample/fill masked tokens, then remask.
    Supports: "random" (Algorithm 4 in LLaDA), "low_confidence" (Alg.5 in LLaDA), and the semi-AR
    variants "block" (left-to-right prefix reveal) and "block_lowconf" (confidence-ranked reveal within the next block).
    Optional unsupervised CFG via `guidance_strength` by masking the prompt for the unconditional branch.

    NOTE: batch size must be 1. This sampler indexes x_t[0] and returns shape [1, L].
    """
    tf.debugging.assert_equal(tf.shape(x_t)[0], 1, message="llada_reverse_step expects batch size 1")
    t = tf.cast(step, tf.float32) / float(num_steps)
    s = tf.cast(step - 1, tf.float32) / float(num_steps)
    L = tf.shape(x_t)[1]
    M = tf.cast(mask_token_id, x_t.dtype)

    # 1) Fill masked tokens with optional CFG and optional temperature sampling
    row_ids = tf.range(L, dtype=tf.int32)

    logits_c = model(x_t, training=False)[0]                                   # [L,V]
    logits_c = apply_carry_over_logits(logits_c[None, ...], x_t, mask_token_id)[0]
    # Confidence for the actually chosen ids is computed after sampling below.

    if guidance_strength and guidance_strength > 0.0:
        # Prompt-only unconditional branch (leave suffix as-is)
        u_mask = tf.concat([tf.ones([prompt_len], dtype=tf.bool), tf.zeros([L - prompt_len], dtype=tf.bool)], axis=0)
        x_uncond = tf.where(u_mask, tf.fill([L], M), x_t[0])[None, :]
        logits_u = model(x_uncond, training=False)[0]
        logits_u = apply_carry_over_logits(logits_u[None, ...], x_uncond, mask_token_id)[0]
        logits = _cfg_combine(logits_c, logits_u, guidance_strength)
        # CFG logits at prompt positions are not used for token replacement or remasking:
        # force-copy preserves the prompt and prompt confidence is excluded below.
    else:
        logits = logits_c

    # Mask indicators (used by repeat penalty and Gumbel temperature sampling)
    mask = tf.equal(x_t[0], M)                                                # [L]
    mask_f = tf.cast(mask, logits.dtype)[..., None]                           # [L,1]

    # discourage immediate repeats at masked positions
    left = tf.concat([tf.fill([1], tf.cast(model.config.eos_token_id, x_t.dtype)), x_t[0][:-1]], axis=0)  # [L]
    left_is_real = tf.not_equal(left, tf.cast(M, left.dtype))  # only if left token isn't MASK
    repeat_onehot = tf.one_hot(tf.cast(left, tf.int32), tf.shape(logits)[-1], dtype=logits.dtype)  # [L,V]
    repeat_onehot = repeat_onehot * tf.cast(left_is_real, logits.dtype)[..., None] * mask_f  # only apply on masked positions
    logits = logits - tf.cast(repeat_penalty, logits.dtype) * repeat_onehot

    # Gumbel-max sampling in fp32. argmax(logits + tau*Gumbel) samples from softmax(logits/tau).
    # tau=0 gives pure greedy sampling (the reference-style setting).
    u32 = tf.random.uniform(tf.shape(logits), minval=1e-6, maxval=1.0-1e-6, dtype=tf.float32)
    g32 = -tf.math.log(-tf.math.log(u32))
    tau32 = tf.maximum(tf.cast(tau_end + (tau_start - tau_end) * t, tf.float32), 0.0)
    jitter32 = tf.cast(mask_f, tf.float32) * (tau32 * g32)
    logits_sample = logits + tf.cast(jitter32, logits.dtype)
    pred_ids = tf.argmax(logits_sample, axis=-1, output_type=x_t.dtype)       # [L]

    # For confidence, score under the *base* logits (no jitter), gathered at chosen ids.
    probs = tf.nn.softmax(tf.cast(logits, tf.float32), axis=-1)
    conf = tf.gather_nd(probs, tf.stack([row_ids, tf.cast(pred_ids, tf.int32)], axis=1))  # [L]

    filled = tf.where(mask, pred_ids, x_t[0])

    # 2) Remask per strategy
    if strategy == "random":
        # Alg.4: only positions that were MASK at t can remain MASK with prob s/t
        p_stay = tf.clip_by_value(s / tf.maximum(t, 1e-6), 0.0, 1.0)
        u = tf.random.uniform([L], dtype=tf.float32)
        stay_mask = tf.logical_and(mask, u < p_stay)
        x_s = tf.where(stay_mask, tf.fill([L], M), filled)
        return x_s[None, :]

    if strategy in ("block", "block_lowconf"):
        # Reveal at most `block_size` new tokens beyond the previous prefix.
        target_nun = tf.cast(tf.floor(tf.cast(L - prompt_len, tf.float32) * (1.0 - s)), tf.int32) + tf.cast(prompt_len, tf.int32)
        target_nun = tf.clip_by_value(target_nun, 0, L)
        prev_nun   = tf.cast(tf.floor(tf.cast(L - prompt_len, tf.float32) * (1.0 - t)), tf.int32) + tf.cast(prompt_len, tf.int32)
        prev_nun   = tf.clip_by_value(prev_nun, 0, L)
        nun        = tf.minimum(target_nun, prev_nun + tf.cast(block_size, tf.int32))

        # prefix we keep unmasked after this step
        keep_prefix = tf.concat([
            tf.ones([prompt_len], dtype=tf.bool),
            tf.sequence_mask(nun - prompt_len, maxlen=tf.cast(L - prompt_len, tf.int32))
        ], axis=0)

        if strategy == "block":
            x_s = tf.where(keep_prefix, filled, tf.fill([L], M))
            return x_s[None, :]

        # Confidence-ranked local block reveal. Preserve every token already revealed,
        # then reveal only the highest-confidence masked tokens from the next local block.
        current_keep = tf.logical_not(mask)
        current_nun = tf.reduce_sum(tf.cast(current_keep, tf.int32))
        need = tf.maximum(nun - current_nun, 0)
        suffix_mask = tf.logical_and(mask, tf.range(L, dtype=tf.int32) >= tf.cast(prompt_len, tf.int32))
        masked_idx = tf.cast(tf.where(suffix_mask)[:, 0], tf.int32)
        n_candidates = tf.minimum(tf.shape(masked_idx)[0], tf.cast(block_size, tf.int32))
        candidate_idx = masked_idx[:n_candidates]
        candidate_conf = tf.gather(conf, candidate_idx)
        candidate_order = tf.argsort(candidate_conf, direction="DESCENDING")
        take = tf.minimum(need, n_candidates)
        add_idx = tf.gather(candidate_idx, candidate_order[:take])

        def _reveal():
            final_keep = tf.tensor_scatter_nd_update(current_keep, tf.expand_dims(add_idx, 1), tf.fill([take], True))
            return tf.where(final_keep, filled, tf.fill([L], M))
        x_s = tf.cond(take > 0, _reveal, lambda: tf.where(current_keep, filled, tf.fill([L], M)))
        return x_s[None, :]

    # default: low_confidence. Only positions masked at the current step can be remasked;
    # prompt and previously revealed tokens are preserved.
    candidate = mask

    # Use confidences at chosen ids; non-candidates (prompt and previously revealed tokens) get +inf so they are never remasked
    conf_adj = tf.where(candidate, conf, tf.fill([L], tf.constant(1e9, conf.dtype)))

    # Target number unmasked after this step
    nun = tf.cast(tf.floor(tf.cast(L - prompt_len, tf.float32) * (1.0 - s)), tf.int32) + tf.cast(prompt_len, tf.int32)

    k_remask = tf.maximum(L - nun, 0)
    order = tf.argsort(conf_adj, direction="ASCENDING")
    k = tf.cast(tf.maximum(k_remask, 0), tf.int32)
    to_remask = tf.gather(order, tf.range(k))

    def _updated():
        return tf.tensor_scatter_nd_update(
            filled,
            indices=tf.expand_dims(to_remask, 1),
            updates=tf.where(
                tf.gather(candidate, to_remask),
                tf.fill([k], M),
                tf.gather(filled, to_remask)
            )
        )

    x_s = tf.cond(k > 0, _updated, lambda: filled)
    return x_s[None, :]

# To standardize performance metrics: pseudo-perplexity (PPP) via leave-one-out masking (Salazar et al., 2020)
def evaluate_ppp(model, dataset, positions_per_seq=64, microbatch=256):
    """
    Low-memory PPP: project logits only at masked positions, in chunks.
    """
    mask_id = tf.cast(model.config.mask_token_id, tf.int32)

    total_nats = 0.0
    total_cnt  = 0

    for x0 in dataset:
        B = _to_python_int(tf.shape(x0)[0])
        T = _to_python_int(tf.shape(x0)[1])
        m = min(int(positions_per_seq), T)

        # sample m distinct positions per sequence (no replacement)
        perm = tf.argsort(tf.random.uniform([B, T], dtype=tf.float32), axis=1)
        idx = perm[:, :m]  # [B, m]

        # tile inputs to [B*m, T]
        x_tiled   = tf.repeat(x0, repeats=m, axis=0)
        flat_idx  = tf.reshape(idx, [-1])                         # [B*m]
        row_idx   = tf.range(tf.shape(x_tiled)[0], dtype=tf.int32)

        # targets at masked positions
        flat_targ = tf.gather_nd(x_tiled, tf.stack([row_idx, flat_idx], axis=1))

        # mask exactly that one pos per row
        xt = tf.tensor_scatter_nd_update(
            x_tiled,
            tf.stack([row_idx, flat_idx], axis=1),
            tf.fill([tf.shape(x_tiled)[0]], mask_id),
        )

        # chunk to keep memory small
        N = _to_python_int(tf.shape(xt)[0])
        start = 0
        while start < N:
            end = min(start + int(microbatch), N)
            xt_c     = xt[start:end]
            pos_c    = flat_idx[start:end]
            targ_c   = flat_targ[start:end]

            h = model.encode(xt_c, training=False)                     # [n, T, C]
            logits_sel = model.logits_at_positions(h, pos_c)           # [n, V]

            nll = keras.losses.sparse_categorical_crossentropy(
                targ_c, tf.cast(logits_sel, tf.float32), from_logits=True
            )
            total_nats += _to_python_float(tf.reduce_sum(nll))
            total_cnt  += _to_python_int(tf.shape(nll)[0])
            start = end

    nats_per_tok = total_nats / max(total_cnt, 1)
    bpt = nats_per_tok / float(np.log(2.0))
    ppl = float(np.exp(nats_per_tok))
    return bpt, ppl

def llada_generate(
    model, tokenizer, prompt, total_len, steps=None, strategy="low_confidence",
    guidance_strength=None, block_size=None, tau_start=None, tau_end=None, repeat_penalty=None,
):
    """Generate by iterative masked denoising while keeping prompt tokens fixed."""
    total_len = min(int(total_len), int(model.config.context_length))
    cfg = model.config
    steps = cfg.num_sampling_steps if steps is None else int(steps)
    M, eos = cfg.mask_token_id, cfg.eos_token_id
    g = cfg.guidance_strength if guidance_strength is None else guidance_strength
    b = cfg.block_size if block_size is None else block_size
    tau_start = cfg.sampling_tau_start if tau_start is None else tau_start
    tau_end = cfg.sampling_tau_end if tau_end is None else tau_end
    repeat_penalty = cfg.sampling_repeat_penalty if repeat_penalty is None else repeat_penalty

    p_ids = tokenizer.encode(prompt)[:total_len]
    L, P = total_len, len(p_ids)
    x = tf.constant([p_ids + [M] * (L - P)], dtype=tf.int32)

    # Notebook-specific CFG bootstrap: keep guidance off for a few initial all/mostly-masked steps.
    bootstrap_steps = int(min(6, max(1, L - P)))
    for step in range(steps, 0, -1):
        use_g = 0.0 if step > steps - bootstrap_steps else g
        x = llada_reverse_step(
            model, x, step, steps, mask_token_id=M, prompt_len=P, strategy=strategy,
            guidance_strength=use_g, block_size=b, tau_start=tau_start, tau_end=tau_end,
            repeat_penalty=repeat_penalty,
        )

    ids = [token_id for token_id in x.numpy()[0].tolist() if token_id != M]
    if eos in ids[P:]:
        ids = ids[:ids.index(eos, P) + 1]
    return tokenizer.decode(ids)


class WarmupHoldLinearTransition(tf.keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear transition from `start` to `end` during the ramp phase.
    Before the ramp: constant = start. After the ramp: constant = end.
    """
    def __init__(self, start, end, warmup_steps, hold_steps, ramp_steps):
        self.start = tf.convert_to_tensor(start, tf.float32)
        self.end   = tf.convert_to_tensor(end,   tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.hold_steps   = tf.cast(hold_steps,   tf.float32)
        self.ramp_steps   = tf.cast(tf.maximum(ramp_steps, 1), tf.float32)

    def __call__(self, step):
        s = tf.cast(step, tf.float32)
        # offset into the ramp portion (0 before ramp starts)
        after_wh = tf.maximum(s - (self.warmup_steps + self.hold_steps), 0.0)
        frac = tf.clip_by_value(after_wh / self.ramp_steps, 0.0, 1.0)
        return self.start + (self.end - self.start) * frac

    def get_config(self):
        return {
            "start": float(self.start.numpy()),
            "end": float(self.end.numpy()),
            "warmup_steps": float(self.warmup_steps.numpy()),
            "hold_steps": float(self.hold_steps.numpy()),
            "ramp_steps": float(self.ramp_steps.numpy()),
        }



## 5. Evaluation metrics

This notebook reports:

- **Masked BPT**: bits per token at a fixed corruption rate.
- **Masked PPL @ t=0.15**: `exp(masked CE)` at fixed 15% corruption.
- **PPP**: pseudo-perplexity from leave-one-out masking.

These are **not directly comparable to ordinary autoregressive perplexity** because the bidirectional model can see both left and right context.

The final evaluation also reports `BPT @ t=1.0`, a useful diagnostic because generation begins from a fully masked response.


In [ ]:
# ----------------------- Bits-per-token / NLL-style eval --------------------
def evaluate_bpt(model, dataset, t_eval=0.5, n_mc=4, microbatch_pos=2048):
    """
    Bits-per-token at a fixed corruption rate using exact-k masking, without ever
    materializing [B, T, V]. Projects only the masked positions.

    Args:
        model: LLaDAModel with .encode(), .embed.tok_emb.embeddings, .head_bias.
        dataset: tf.data.Dataset of integer token windows shape [B, T].
        t_eval: mask ratio for evaluation (clipped to [0.05, 1.0]).
        n_mc: Monte Carlo repeats per batch (averaged).
        microbatch_pos: max masked positions per projection chunk.

    Returns:
        Scalar float: average bits/token over the dataset.
    """
    t_eval = float(np.clip(t_eval, 0.05, 1.0))
    log2e = 1.0 / float(np.log(2.0))
    mask_id = tf.cast(model.config.mask_token_id, tf.int32)

    def _bpt_for_batch(x0):
        B = _to_python_int(tf.shape(x0)[0])
        acc_ce = 0.0

        for _ in range(max(1, n_mc)):
            # Exact-k masking at fixed t_eval
            t_vec = tf.fill([B], tf.cast(t_eval, tf.float32))
            xt, mask_bool, _ = mask_exact_k_given_t(x0, mask_id, t_vec)  # [B,T], [B,T]

            # Encode once for the whole batch
            h = model.encode(xt, training=False)  # [B,T,C]

            # Precompute projection params (once per MC sample)
            E  = model.embed.tok_emb.embeddings
            hb = tf.cast(model.head_bias, E.dtype)

            # Indices of masked positions
            pos = tf.where(mask_bool)                # [M,2] with (row,col)
            row_idx = tf.cast(pos[:, 0], tf.int32)
            col_idx = tf.cast(pos[:, 1], tf.int32)

            # Targets at those positions
            targ = tf.gather_nd(x0, pos)            # [M]

            # Chunk over masked positions to limit memory
            M = _to_python_int(tf.shape(pos)[0])
            start = 0
            ce_list = []

            while start < M:
                end = min(start + int(microbatch_pos), M)
                rows_c = row_idx[start:end]
                cols_c = col_idx[start:end]

                # Gather the hidden states for just these positions and project
                h_sel = tf.gather_nd(h, tf.stack([rows_c, cols_c], axis=1))  # [m,C]
                logitsH = tf.matmul(h_sel, tf.cast(E, h_sel.dtype), transpose_b=True) + tf.cast(hb, h_sel.dtype)
                logitsH = model._forbid_mask(logitsH)

                targ_c = targ[start:end]
                nll = keras.losses.sparse_categorical_crossentropy(targ_c, tf.cast(logitsH, tf.float32), from_logits=True)
                ce_list.append(nll)
                start = end

            if ce_list:
                ce_all = tf.concat(ce_list, axis=0)                          # [M]
                # Mean CE per example over its masked positions
                ce_sum = tf.math.unsorted_segment_sum(ce_all, row_idx, num_segments=B)
                cnt    = tf.math.unsorted_segment_sum(tf.ones_like(ce_all), row_idx, num_segments=B)
                ce_masked = ce_sum / tf.maximum(cnt, 1.0)                    # [B] nats/token
                acc_ce += _to_python_float(tf.reduce_mean(ce_masked))
            else:
                # Degenerate (shouldn't happen with exact-k), but keep safe
                acc_ce += 0.0

        return (acc_ce / float(max(1, n_mc))) * log2e  # bits/token

    # Dataset average (weighted by batch size)
    total_bpt, total_ex = 0.0, 0
    for x0 in dataset:
        B = _to_python_int(tf.shape(x0)[0])
        total_bpt += _bpt_for_batch(x0) * B
        total_ex  += B

    return total_bpt / max(1, total_ex)


# ---- helpers used by the loop ------------------------------------------------
# Gradient clipping is handled by the optimizer's `global_clipnorm`.


## 6. Sparse-projection training and validation steps

A major memory-saving feature is that training projects hidden states to vocabulary logits **only at masked positions**, instead of materializing a dense `[batch, sequence, vocabulary]` tensor.

The optional repetition regularizer is explicitly non-LLaDA. Its global scale decays to zero early in training.


In [ ]:
@tf.function(jit_compile=False)
def _train_step_llada(model, optimizer, x0, mask_token_id, t_max=1.0, repeat_scale=0.0):
    B = tf.shape(x0)[0]; T = tf.shape(x0)[1]

    # ---- sample t per example (adaptive bounds)
    t_max = tf.cast(t_max, tf.float32)
    t_min = tf.minimum(t_max * 0.5, 0.25); t_min = tf.maximum(t_min, 0.05)
    t_vec = sample_t_stratified(B, t_min=t_min, t_max=t_max)

    # ---- Bernoulli per-token masking
    xt, mask_bool, _ = mask_with_given_t(x0, mask_token_id, t_vec)

    m = tf.cast(mask_bool, tf.float32)             # [B,T]
    cnt_per_ex = tf.reduce_sum(m, axis=1)          # [B]
    total_cnt  = tf.reduce_sum(cnt_per_ex)         # scalar

    # ---- repetition loss knobs (fallbacks if not on config)
    cfg = model.config
    rep_w_immediate = tf.constant(float(getattr(cfg, "rep_w_immediate", 0.25)), tf.float32)
    rep_w_two_back  = tf.constant(float(getattr(cfg, "rep_w_two_back",  0.10)), tf.float32)
    rep_w_presence  = tf.constant(float(getattr(cfg, "rep_w_presence",  0.05)), tf.float32)
    rep_win         = tf.constant(int(getattr(cfg, "rep_presence_window", 32)), tf.int32)
    rep_k           = tf.constant(int(getattr(cfg, "rep_presence_k", 8)), tf.int32)

    W = tf.minimum(T, rep_win)
    K = tf.minimum(W, rep_k)
    presence_enabled = tf.logical_and(tf.greater(rep_w_presence, 0.0), tf.greater(K, 0))

    # Gate repetition penalties by runtime repeat_scale and configured weights.
    repeat_scale = tf.cast(repeat_scale, tf.float32)
    any_rep_w = tf.logical_or(
        tf.greater(rep_w_immediate, 0.0),
        tf.logical_or(
            tf.greater(rep_w_two_back, 0.0),
            tf.logical_and(tf.greater(rep_w_presence, 0.0), tf.greater(K, 0)),
        ),
    )
    do_rep = tf.logical_and(tf.greater(repeat_scale, 0.0), any_rep_w)

    eos_id = tf.cast(cfg.eos_token_id, x0.dtype)

    with tf.GradientTape() as tape:
        # ---- encode once
        h = model.encode(xt, training=True)  # [B,T,C]

        # ---- projection params (variables stored in fp32; cast to the compute dtype at use)
        E   = model.embed.tok_emb.embeddings                              # [V,C], fp32 variable
        hb  = tf.cast(model.head_bias, E.dtype)                           # [V]
        mb  = tf.cast(model._mask_bias, E.dtype)                          # [V]

        # ---- masked positions
        # The main memory saving comes from projecting only masked positions instead of
        # dense [B,T,V] logits. Chunking limits each forward projection tensor, although
        # tensors required for the backward pass may remain live under GradientTape.
        pos = tf.where(mask_bool)                                      # [M,2]
        row_idx = tf.cast(pos[:, 0], tf.int32)
        col_idx = tf.cast(pos[:, 1], tf.int32)
        M = tf.shape(row_idx)[0]

        # ---- accumulators (per-example sums)
        ce_sum  = tf.zeros([B], tf.float32)
        t1_sum  = tf.zeros([B], tf.float32)   # immediate repeat P(y==left)
        t2_sum  = tf.zeros([B], tf.float32)   # two-back repeat P(y==left2)
        t3_sum  = tf.zeros([B], tf.float32)   # presence mean P over candidate set

        chunk = tf.constant(128, tf.int32)  # smaller [m,V] slices to reduce peak
        
        def cond(start, ce_acc, a1, a2, a3):
            return start < M

        def body(start, ce_acc, a1, a2, a3):
            end    = tf.minimum(start + chunk, M)
            rows_c = row_idx[start:end]                                     # [m]
            cols_c = col_idx[start:end]                                     # [m]

            # gather hidden states at masked positions, project to logits
            h_sel   = tf.gather_nd(h, tf.stack([rows_c, cols_c], axis=1))     # [m, C] in compute dtype
            logitsH = tf.matmul(h_sel, tf.cast(E, h_sel.dtype), transpose_b=True) + tf.cast(hb, h_sel.dtype)
            logitsH = logitsH + tf.cast(mb, logitsH.dtype)
            logitsF = tf.cast(logitsH, tf.float32)                            # upcast only for loss/stats

            # CE on true targets
            targ_c = tf.gather_nd(x0, tf.stack([rows_c, cols_c], axis=1))   # [m]
            nll_c  = keras.losses.sparse_categorical_crossentropy(targ_c, logitsF, from_logits=True)
            ce_acc = ce_acc + tf.math.unsorted_segment_sum(nll_c, rows_c, num_segments=B)

            # --- repetition penalties (gated; no full softmax) --------------------------
            def _rep_on(a1, a2, a3):
                # local row indices within this chunk
                rloc = tf.range(tf.shape(rows_c)[0], dtype=tf.int32)                 # [m]
                lse  = tf.reduce_logsumexp(logitsF, axis=-1)                         # [m]

                # Immediate repeat P(y == left)
                prev_cols  = tf.maximum(cols_c - 1, 0)
                left_raw   = tf.gather_nd(x0, tf.stack([rows_c, prev_cols], axis=1)) # [m]
                left_ids   = tf.where(cols_c > 0, left_raw, tf.fill(tf.shape(prev_cols), eos_id))
                left_logits = tf.gather_nd(logitsF, tf.stack([rloc, tf.cast(left_ids, tf.int32)], axis=1))
                p_left      = tf.exp(left_logits - lse)                               # [m]
                a1 = a1 + tf.math.unsorted_segment_sum(p_left, rows_c, num_segments=B)

                # Two-back repeat
                prev2_cols   = tf.maximum(cols_c - 2, 0)
                left2_raw    = tf.gather_nd(x0, tf.stack([rows_c, prev2_cols], axis=1))
                left2_ids    = tf.where(cols_c > 1, left2_raw, tf.fill(tf.shape(prev2_cols), eos_id))
                left2_logits = tf.gather_nd(logitsF, tf.stack([rloc, tf.cast(left2_ids, tf.int32)], axis=1))
                p_left2      = tf.exp(left2_logits - lse)
                a2 = a2 + tf.math.unsorted_segment_sum(p_left2, rows_c, num_segments=B)

                # Presence penalty over up to K tokens immediately preceding each masked position.
                def _presence_on(a3):
                    offsets = tf.range(1, K + 1, dtype=tf.int32)[None, :]             # [1,K]
                    cand_cols = cols_c[:, None] - offsets                            # [m,K]
                    recent_start = tf.maximum(cols_c[:, None] - W, 0)
                    cand_valid = tf.logical_and(cand_cols >= recent_start, cand_cols >= 0)
                    safe_cols = tf.maximum(cand_cols, 0)
                    batch_rows = tf.broadcast_to(rows_c[:, None], tf.shape(safe_cols))
                    cand_ids = tf.gather_nd(x0, tf.stack([batch_rows, safe_cols], axis=-1))
                    rlocK = tf.broadcast_to(rloc[:, None], tf.shape(safe_cols))
                    cand_logits = tf.gather_nd(logitsF, tf.stack([rlocK, tf.cast(cand_ids, tf.int32)], axis=-1))
                    p_cand = tf.exp(cand_logits - lse[:, None])
                    valid_f = tf.cast(cand_valid, tf.float32)
                    p_mean = tf.reduce_sum(p_cand * valid_f, axis=-1) / tf.maximum(tf.reduce_sum(valid_f, axis=-1), 1.0)
                    return a3 + tf.math.unsorted_segment_sum(p_mean, rows_c, num_segments=B)
                a3 = tf.cond(presence_enabled, lambda: _presence_on(a3), lambda: a3)
                return a1, a2, a3

            # Only compute repetition when enabled and when repeat_scale > 0
            a1, a2, a3 = tf.cond(do_rep, lambda: _rep_on(a1, a2, a3), lambda: (a1, a2, a3))

            return end, ce_acc, a1, a2, a3

        _, ce_sum, t1_sum, t2_sum, t3_sum = tf.while_loop(cond, body, loop_vars=[tf.constant(0, tf.int32), ce_sum, t1_sum, t2_sum, t3_sum], parallel_iterations=1)

        # ---- base masked-CE objective (per-example mean)
        per_ex     = ce_sum / tf.maximum(cnt_per_ex, 1.0)       # [B]
        loss_llada = tf.reduce_mean(per_ex)                     # scalar

        # ---- repetition regularizer (per-example mean, then weighted)
        t1_mean = tf.reduce_mean(t1_sum / tf.maximum(cnt_per_ex, 1.0))
        t2_mean = tf.reduce_mean(t2_sum / tf.maximum(cnt_per_ex, 1.0))
        t3_mean = tf.reduce_mean(t3_sum / tf.maximum(cnt_per_ex, 1.0))
        rep_loss = rep_w_immediate * t1_mean + rep_w_two_back * t2_mean + rep_w_presence * t3_mean

        # ---- add any auxiliary model losses (e.g., WD)
        if model.losses:
            loss_llada = loss_llada + tf.add_n([tf.cast(l, tf.float32) for l in model.losses])

        # ---- total train loss for backprop
        loss_total = loss_llada + tf.cast(repeat_scale, rep_loss.dtype) * rep_loss  # auxiliary repetition term decays to zero
        if hasattr(optimizer, "scale_loss"):
            loss_for_grad = optimizer.scale_loss(loss_total)  # Keras 3
        elif hasattr(optimizer, "get_scaled_loss"):
            loss_for_grad = optimizer.get_scaled_loss(loss_total)  # legacy tf.keras
        else:
            loss_for_grad = loss_total

    grads = tape.gradient(loss_for_grad, model.trainable_variables)
    if hasattr(optimizer, "get_unscaled_gradients"):
        grads = optimizer.get_unscaled_gradients(grads)
    # Do not sanitize nonfinite gradients. LossScaleOptimizer must see Inf/NaN
    # gradients so it can skip the update and lower the dynamic loss scale.
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    # ---- "plain" masked CE for logging (same averaging as before)
    loss_plain = tf.reduce_sum(ce_sum) / tf.maximum(total_cnt, 1.0)

    # return CE-based train loss (for continuity), plus plain CE and t schedule
    return tf.cast(loss_llada, tf.float32), tf.cast(loss_plain, tf.float32), t_vec

# Validation metric that matches the (moving-curriculum) training objective
@tf.function(jit_compile=False)
def _valid_step_llada_curriculum(model, x0, mask_token_id, n_samples=4, t_max=1.0):
    """Average the masked-token CE over multiple Bernoulli t samples; matches the train objective."""
    total_loss = 0.0
    for _ in tf.range(n_samples):
        B = tf.shape(x0)[0]
        t_max_now = tf.cast(t_max, tf.float32)
        t_min = tf.minimum(t_max_now * 0.5, 0.25)
        t_min = tf.maximum(t_min, 0.05)
        t_vec = sample_t_stratified(B, t_min=t_min, t_max=t_max_now)
        xt, mask_bool, _ = mask_with_given_t(x0, mask_token_id, t_vec)  # same corruption distribution as training

        # Sparse projection at masked positions (matches training objective but is cheaper)
        h = model.encode(xt, training=False)  # [B,T,C]
        E  = model.embed.tok_emb.embeddings
        hb = tf.cast(model.head_bias, E.dtype)

        pos     = tf.where(mask_bool)                      # [M,2] (row,col)
        rows_c  = tf.cast(pos[:, 0], tf.int32)
        cols_c  = tf.cast(pos[:, 1], tf.int32)
        targ_c  = tf.gather_nd(x0, pos)                    # [M]

        h_sel   = tf.gather_nd(h, pos)                     # [M,C]
        logitsH = tf.matmul(h_sel, tf.cast(E, h_sel.dtype), transpose_b=True) + tf.cast(hb, h_sel.dtype)
        logitsH = model._forbid_mask(logitsH)
        nll = keras.losses.sparse_categorical_crossentropy(targ_c, tf.cast(logitsH, tf.float32), from_logits=True)

        # Average per-example over its masked positions, then batch-mean
        ce_sum = tf.math.unsorted_segment_sum(nll, rows_c, num_segments=tf.shape(x0)[0])
        cnt    = tf.math.unsorted_segment_sum(tf.ones_like(nll), rows_c, num_segments=tf.shape(x0)[0])
        total_loss += tf.reduce_mean(ce_sum / tf.maximum(cnt, 1.0))

    return total_loss / tf.cast(n_samples, tf.float32)

@tf.function(jit_compile=False)
def _valid_step_llada_fixed_t(model, x0, mask_token_id, t_fixed=0.15, microbatch_pos=4096):
    """
    Low-memory fixed-t CE: project only masked positions in chunks.
    """
    B = tf.shape(x0)[0]
    t_vec = tf.fill([B], tf.cast(t_fixed, tf.float32))
    xt, mask_bool, _ = mask_exact_k_given_t(x0, mask_token_id, t_vec)

    h = model.encode(xt, training=False)  # [B,T,C]
    E  = model.embed.tok_emb.embeddings
    hb = tf.cast(model.head_bias, E.dtype)

    pos = tf.where(mask_bool)                   # [M,2]
    row_idx = tf.cast(pos[:, 0], tf.int32)
    col_idx = tf.cast(pos[:, 1], tf.int32)
    targ    = tf.gather_nd(x0, pos)            # [M]

    M = tf.shape(row_idx)[0]
    start = tf.constant(0, tf.int32)
    ce_acc = tf.zeros([B], tf.float32)

    def cond(s, acc):
        return s < M

    def body(s, acc):
        end    = tf.minimum(s + tf.cast(microbatch_pos, tf.int32), M)
        rows_c = row_idx[s:end]; cols_c = col_idx[s:end]
        h_sel  = tf.gather_nd(h, tf.stack([rows_c, cols_c], axis=1))  # [m,C]
        logits = tf.matmul(h_sel, tf.cast(E, h_sel.dtype), transpose_b=True) + tf.cast(hb, h_sel.dtype)
        logits = model._forbid_mask(logits)
        nll    = keras.losses.sparse_categorical_crossentropy(targ[s:end], tf.cast(logits, tf.float32), from_logits=True)
        acc    = acc + tf.math.unsorted_segment_sum(nll, rows_c, num_segments=B)
        return end, acc

    _, ce_sum = tf.while_loop(cond, body, [start, ce_acc], parallel_iterations=1)
    cnt = tf.math.unsorted_segment_sum(tf.ones_like(row_idx, tf.float32), row_idx, num_segments=B)
    return tf.reduce_mean(ce_sum / tf.maximum(cnt, 1.0))



## 7. Training loop

A **training round** is a capped number of optimization steps, not necessarily a full pass over WikiText-103. The rotating global permutation gives broad corpus coverage over successive rounds.

The corruption ceiling ramps from `0.20` to `1.0`. Because the training corruption distribution changes, the moving-curriculum loss can rise even while the fixed-corruption metric improves.

For notebook observability, the loop prints progress during optimization and announces validation / preview generation explicitly. **Final generation, masked-BPT evaluation, and PPP are not hidden inside training**; they are separate runnable cells later in the notebook.


In [ ]:
# ----------------------- LLaDA-style training loop ----------------------------
def train_diffusion(
    cfg,
    val_ds=None,
    rounds=8,
    base_learning_rate=1e-4,
    weight_decay=0.0,
    clipnorm=4.0,
    patience=5,
    min_delta=0.0,
    plot_every=1,
    sample_prompt="In a distant library,",
    sample_strategy="low_confidence",
    sample_truth_text: str | None = None,
    max_train_steps_per_round: int | None = 4000,   
    X_train: np.ndarray | None = None,  
    batch_size: int | None = None,
    output_dir: str = "outputs",
    tokenizer=None,
    progress_every_steps: int = 25,
):
    """
    Discrete diffusion training with fixed-corruption evaluation and EMA checkpoints.
    - Tracks train/val loss and derives perplexity from plain masked CE
    - Warmup-Hold-Decay LR schedule and per-step LR logging
    - Early stopping on fixed validation CE at t=0.15
    - One-panel plot (loss + PPL) and periodic preview samples
    - Visible stage/progress messages so long operations are diagnosable in notebooks
    """
    # ----- setup
    tf.random.set_seed(123); np.random.seed(123)
    os.makedirs(output_dir, exist_ok=True)
    raw_ckpt_path = os.path.join(output_dir, "model_tf_diffusion.weights.h5")
    ema_ckpt_path = os.path.join(output_dir, "model_tf_diffusion_ema.weights.h5")
    config_path = os.path.join(output_dir, "model_config.json")
    model = LLaDAModel(cfg)
    _ = model(tf.zeros((1, cfg.context_length), dtype=tf.int32), training=False)

    # ----- uniform rotating data subset setup
    if X_train is None or batch_size is None:
        raise RuntimeError("Provide X_train (numpy windows) and batch_size for rotating sampling.")
    N = int(len(X_train))
    if N == 0:
        raise RuntimeError("Empty X_train.")
    # Full-sweep batch count (drop_remainder=True like before)
    num_full_batches = N // batch_size
    if num_full_batches == 0:
        raise RuntimeError("batch_size larger than dataset.")
    # If you cap steps, we’ll cover a *contiguous slice* of a single global permutation each training round,
    # then continue from where we left off; on wrap, reshuffle.
    if max_train_steps_per_round is None:
        steps_per_round = num_full_batches
    else:
        steps_per_round = max(1, min(int(max_train_steps_per_round), num_full_batches))
    M = steps_per_round * batch_size  # number of examples consumed per training round
    rng = np.random.default_rng(123)
    order = np.arange(N)
    rng.shuffle(order)
    cursor = 0

    # Fixed mini train-eval split for stable train-side metric (optional)
    _eval_batches = 4
    _eval_size = _eval_batches * batch_size
    eval_idx = order[:_eval_size]  # taken from the initial permutation for reproducibility
    train_eval_ds = (tf.data.Dataset.from_tensor_slices(X_train[eval_idx]).batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE))

    mask_const = tf.constant(cfg.mask_token_id, dtype=tf.int32)
    total_steps = max(rounds * steps_per_round, 1)

    # Repetition-loss schedule: hold (rep_peak) to 1% (of max training), linearly ramp to 0 by 31%, then stay 0
    rep_peak = float(cfg.rep_schedule_peak)
    rep_sched = WarmupHoldLinearTransition(
        start=rep_peak,                           # constant before ramp
        end=0.0,                                  # value after ramp
        warmup_steps=int(0.010 * total_steps),     # 0-1%: hold rep_peak
        hold_steps=0,
        ramp_steps=max(1, int(0.30 * total_steps))# 1-31%: ramp → 0; ≥31% stays 0
    )

    # Step count-based Learning Rate
    lr_peak_round=3
    lr_hold_rounds=0
    lr_min_ratio: float = 1/10
    peak_after_steps = int(round(lr_peak_round * steps_per_round))
    hold_steps       = int(round(lr_hold_rounds * steps_per_round))
    warmup = max(1, min(peak_after_steps, total_steps - 1))
    hold   = max(0, min(hold_steps,    max(total_steps - warmup - 1, 0)))
    decay  = max(1, total_steps - warmup - hold)

    # Masking schedule (shares learning-schedule phase boundaries): start easy and ramp harder across training
    tmax_start = 0.20
    tmax_final = 1.0  # generation starts fully masked, so the curriculum reaches t=1 by the end
    tmax_sched = WarmupHoldLinearTransition(
        start=tmax_start,
        end=tmax_final,
        warmup_steps=warmup,   # keep it easy during LR warmup
        hold_steps=hold,
        ramp_steps=decay
    )

    # Learning Rate scheduler
    lr_sched = WarmupStableDecay(
        base_lr=base_learning_rate,
        warmup_steps=warmup,
        hold_steps=hold,
        decay_steps=decay,
        min_lr=base_learning_rate * lr_min_ratio,
    )

    optimizer_kwargs = dict(learning_rate=lr_sched, weight_decay=weight_decay, beta_1=0.9, beta_2=0.98, epsilon=1e-8)
    if clipnorm and clipnorm > 0:
        optimizer_kwargs["global_clipnorm"] = float(clipnorm)
    opt_base = keras.optimizers.AdamW(**optimizer_kwargs)
    if hasattr(opt_base, "exclude_from_weight_decay"):
        opt_base.exclude_from_weight_decay(var_names=["bias", "beta", "gamma", "norm", "embedding", "head_bias"])
    opt = mixed_precision.LossScaleOptimizer(opt_base)

    ema = EMA(model, decay=0.999)
    ema_best_shadow = None  # lives in train_diffusion’s closure

    def _load_ema_into(dst_model, prefer_best=False):
        """Load EMA weights into dst_model. For live previews, set prefer_best=False."""
        if prefer_best and (ema_best_shadow is not None):
            for s, v in zip(ema_best_shadow, dst_model.trainable_weights):
                v.assign(s)
        else:
            ema.copy_to(dst_model)

    tok = tokenizer if tokenizer is not None else get_tokenizer()
    sample_model = LLaDAModel(cfg)
    _ = sample_model(tf.zeros((1, cfg.context_length), dtype=tf.int32), training=False)

    # ----- trackers
    train_losses, val_losses = [], []
    train_ppls,   val_ppls   = [], []
    lrs = []
    round_seconds = []
    best_round = None
    best_metric = np.inf  # fixed validation CE at t=0.15
    wait = 0
    global_step = 0

    for round_idx in range(rounds):
        round_t0 = time.perf_counter()
        print(f"\nRound {round_idx+1}/{rounds}: training {steps_per_round} steps...", flush=True)
        if round_idx == 0:
            print("  (the first training steps may be slower while TensorFlow traces the graph)", flush=True)
        # Slice the next round's chunk from a single global permutation (uniform coverage over time)
        if cursor + M < N:
            idx = order[cursor:cursor + M]
            cursor += M
        elif cursor + M == N:
            idx = order[cursor:cursor + M]
            rng.shuffle(order)
            cursor = 0
        else:
            tail = order[cursor:]
            need = M - len(tail)
            rng.shuffle(order)
            idx = np.concatenate([tail, order[:need]])
            cursor = need

        # Build this training round's dataset (shuffle within-round for SGD noise)
        round_ds = tf.data.Dataset.from_tensor_slices(X_train[idx])
        round_ds = round_ds.shuffle(buffer_size=min(M, 10000), reshuffle_each_iteration=True)
        round_ds = round_ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

        trL = 0.0
        for step_in_round, x0 in enumerate(round_ds, start=1):
            tmax_now = tmax_sched(tf.constant(global_step, tf.int64))
            # Repetition-loss weight tied to a linear transition schedule
            repeat_scale = tf.cast(rep_sched(tf.constant(global_step, tf.int64)), tf.float32)
            loss_llada, _loss_plain, _t = _train_step_llada(model, opt, x0, mask_const, t_max=tmax_now, repeat_scale=repeat_scale)
            trL += _to_python_float(loss_llada)
            global_step += 1
            ema.update()
            if progress_every_steps and (step_in_round % progress_every_steps == 0 or step_in_round == steps_per_round):
                print(f"  train step {step_in_round}/{steps_per_round} | loss {(_to_python_float(loss_llada)):.4f}", flush=True)
        trL /= max(steps_per_round, 1)

        # Validate
        print("  validating...", flush=True)
        val_t0 = time.perf_counter()
        has_val = False
        vaL_llada = np.nan
        vaL_plain = np.nan
        eval_model = sample_model # Evaluate with EMA weights so metrics match the sampler
        _load_ema_into(eval_model, prefer_best=False)
        if val_ds is not None:
            val_steps = int(tf.data.experimental.cardinality(val_ds).numpy())
            if val_steps > 0:
                has_val = True
                vaL_llada = 0.0
                vaL_plain = 0.0
                for x0 in val_ds:
                    tmax_now = tmax_sched(tf.constant(global_step, tf.int64))
                    vaL_llada += _to_python_float(_valid_step_llada_curriculum(eval_model, x0, mask_const, n_samples=4, t_max=tmax_now))
                    vaL_plain += _to_python_float(_valid_step_llada_fixed_t(eval_model, x0, mask_const, t_fixed=0.15))
                vaL_llada /= val_steps
                vaL_plain /= val_steps
        print(f"  validation complete in {time.perf_counter() - val_t0:.1f}s", flush=True)

        # To generate a stable train-side metric: a fixed evaluation set once per run and always compute tr_fixed on that
        tr_fixed = 0.0
        for x0_eval in train_eval_ds:
            tr_fixed += _to_python_float(_valid_step_llada_fixed_t(eval_model, x0_eval, mask_const, t_fixed=0.15))
        tr_fixed /= _eval_batches

        # Log LR after round
        lrs.append(_to_python_float(lr_sched(global_step)))

        # Accumulate history (PPL = exp(plain masked CE))
        train_losses.append(trL); val_losses.append(vaL_llada)
        train_ppls.append(math.exp(tr_fixed))
        val_ppls.append(math.exp(vaL_plain) if not np.isnan(vaL_plain) else np.nan)

        # Early stopping on fixed validation CE at t=0.15
        if has_val:
            monitored = vaL_plain  # CE at t=0.15; early-stop on the same fixed-corruption CE used for masked PPL
            improved = (best_metric - monitored > min_delta)
            if improved:
                best_metric = monitored
                best_round = round_idx
                wait = 0
                model.save_weights(raw_ckpt_path)
                with open(config_path, "w") as f:
                    json.dump(asdict(cfg), f)
                ema_best_shadow = [tf.identity(s) for s in ema.shadow]
                try:
                    tmp_ema_model = LLaDAModel(cfg)
                    _ = tmp_ema_model(tf.zeros((1, cfg.context_length), dtype=tf.int32), training=False)
                    for s, v in zip(ema_best_shadow, tmp_ema_model.trainable_weights):
                        v.assign(s)
                    tmp_ema_model.save_weights(ema_ckpt_path)
                except Exception as e:
                    print(f"(Saving EMA weights skipped: {e})")
            else:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping @ round {round_idx+1}; best val CE@t=0.15 "
                        f"{best_metric:.4f} at round {best_round+1 if best_round is not None else 'N/A'}.")
                    break

        # Plot & sample
        if plot_every and ((round_idx + 1) % plot_every == 0 or round_idx == 0):
            try:
                clear_output(wait=True)
                plot_lm_training_one_panel(train_losses, val_losses, train_ppls, val_ppls, lrs, best_round, logy=False)
                print(f"Round {round_idx+1}/{rounds}: generating preview sample...", flush=True)

                _load_ema_into(sample_model, prefer_best=False)
                sample_steps = cfg.num_sampling_steps
                sample_txt = llada_generate(
                    sample_model, tok, prompt=sample_prompt, total_len=cfg.context_length,
                    steps=sample_steps, strategy=sample_strategy, guidance_strength=cfg.guidance_strength
                )
                sample_txt = prettify_text(sample_txt)
                print(f"Round {round_idx+1}/{rounds}  train L {trL:.4f} • ppl@t=0.15 {train_ppls[-1]:.1f} | "
                        f"val L(llada) {vaL_llada:.4f} • ppl@t=0.15 {val_ppls[-1]:.1f} | lr {lrs[-1]:.2e} | repeat_scale {_to_python_float(repeat_scale):.3f}")
                print("\n--- LLaDA sample ---\n", sample_txt.replace("\n", " "), sep="")
                if sample_truth_text is not None:
                    true_full = prettify_text(sample_prompt + sample_truth_text)
                    print("\n--- True text ---\n", true_full, sep="")
            except Exception as e:
                print(f"(Plotting/preview skipped: {e})")

        elapsed_round = time.perf_counter() - round_t0
        round_seconds.append(elapsed_round)
        print(f"Round {round_idx+1} complete in {elapsed_round:.1f}s", flush=True)

    # ==============================  End of training rounds  ==================================
    # Return and evaluate the same best validation EMA state used for generation.
    best_raw_path = raw_ckpt_path
    best_ema_path = ema_ckpt_path
    model_best = LLaDAModel(cfg)
    _ = model_best(tf.zeros((1, cfg.context_length), dtype=tf.int32), training=False)
    if val_ds is not None and os.path.exists(best_ema_path):
        model_best.load_weights(best_ema_path)
        print("Reloaded best EMA checkpoint OK.")
    elif val_ds is not None and os.path.exists(best_raw_path):
        model_best.load_weights(best_raw_path)
        print("Reloaded best raw checkpoint OK (EMA checkpoint unavailable).")
    else:
        ema.copy_to(model_best)
        model_best.save_weights(best_ema_path)
        model.save_weights(best_raw_path)
        with open(config_path, "w") as f:
            json.dump(asdict(cfg), f)

    # Final sampling and expensive metrics are intentionally separate notebook cells.

    history = {
        "train_loss": train_losses, "val_loss": val_losses,
        "train_ppl":  train_ppls,   "val_ppl":  val_ppls,
        "lrs": lrs, "best_round": best_round, "round_seconds": round_seconds,
        "best_metric": best_metric, "monitor": "val_ce_t015",
        "returned_model_state": "best_validation_ema_if_available"
    }
    return model_best, history




## 8. Quick-start and full presets

`quick` is intended for end-to-end code verification and educational experimentation. It uses the smaller 384-dimension / 6-layer / 6-head model, limits training text to 5 million characters, and evaluates on at most 128 deterministically distributed validation windows so evaluation does not dominate the run.

`full` uses the larger 1024-dimension / 12-layer / 16-head model, 180 training rounds, and the full validation split.

> `max_chars` limits how much text is tokenized/trained on **after the dataset is available locally**. On the first run, the dataset loader may still need to download/cache the full WikiText-103 source before applying the quick-start truncation.

> The full preset is a substantial training run. Use `quick` first to verify your environment.


In [ ]:
# ----------------------- Run presets ----------------------------------------
RUN_PRESETS = {
    "quick": {
        "model": dict(emb_dim=384, n_heads=6, num_transformer_layers=6, drop_rate=0.1),
        "max_chars": 5_000_000,
        "stride": 128,
        "batch_size": 16,
        "rounds": 8,
        "max_train_steps_per_round": 250,
        "base_learning_rate": 1.0e-4,
        "patience": 8,
        "output_dir": "outputs/quick",
        "val_max_windows": 128,
        "progress_every_steps": 25,
        "bpt_n_mc": 1,
        "ppp_positions_per_seq": 32,
    },
    "full": {
        "model": dict(emb_dim=1024, n_heads=16, num_transformer_layers=12, drop_rate=0.1),
        "max_chars": None,
        "stride": 128,
        "batch_size": 16,
        "rounds": 180,
        "max_train_steps_per_round": 4000,
        "base_learning_rate": 4.0e-5,
        "patience": 50,
        "output_dir": "outputs/full",
        "val_max_windows": None,
        "progress_every_steps": 250,
        "bpt_n_mc": 4,
        "ppp_positions_per_seq": 128,
    },
}


# Part II — Run the pipeline

## 9. Select a preset and inspect the runtime

Run this first. If no GPU is listed, the model can be dramatically slower because TensorFlow will use float32 CPU execution.


In [ ]:
RUN_PRESET = "quick"  # "quick" or "full"
run = RUN_PRESETS[RUN_PRESET]
cfg = LLaDAConfig(vocab_size=50_258, context_length=128, **run["model"])

gpus = tf.config.list_physical_devices("GPU")
print(f"Preset: {RUN_PRESET}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU(s): {gpus if gpus else 'NONE DETECTED'}")
print(f"Precision policy: {mixed_precision.global_policy().name}")
if not gpus:
    print("WARNING: no GPU detected; even the quick preset may be very slow on CPU.")


## 10. Load WikiText-103

This stage may download/cache WikiText-103 on the first run. The `quick` preset truncates the text **after** the source dataset is available locally.


In [ ]:
stage_t0 = time.perf_counter()
print("Loading WikiText-103...", flush=True)
raw_train, raw_val = load_wikitext_cached(
    dataset_id="wikitext-103-raw-v1",
    local_dir="data/wikitext103",
    force_download=False,
    max_chars=run["max_chars"],
)
print(f"Loaded in {time.perf_counter() - stage_t0:.1f}s")
print(f"Train characters: {len(raw_train):,}")
print(f"Validation characters: {len(raw_val):,}")


## 11. Tokenize

Tokenization is separated from data loading so a slow first run is easy to diagnose.


In [ ]:
stage_t0 = time.perf_counter()
print("Loading GPT-2 tokenizer (may download tokenizer files on the first run)...", flush=True)
tok = get_tokenizer()
print(f"Tokenizer ready in {time.perf_counter() - stage_t0:.1f}s; tokenizing...", flush=True)
ids_tr = encode_in_chunks(raw_train, tok)
ids_va = encode_in_chunks(raw_val, tok)
print(f"Tokenized in {time.perf_counter() - stage_t0:.1f}s")
print(f"Train tokens: {len(ids_tr):,}")
print(f"Validation tokens: {len(ids_va):,}")


## 12. Build training and validation windows

For `quick`, validation is deterministically sampled across the full validation split so recurring validation remains inexpensive.


In [ ]:
stage_t0 = time.perf_counter()
print("Building windows...", flush=True)
Xtr = build_windows(ids_tr, cfg.context_length, run["stride"])
Xva = build_windows(ids_va, cfg.context_length, run["stride"])
val_total_windows = len(Xva)
val_max_windows = run.get("val_max_windows")
if val_max_windows is not None and len(Xva) > val_max_windows:
    val_idx = np.linspace(0, len(Xva) - 1, int(val_max_windows), dtype=np.int64)
    Xva = Xva[val_idx]

batch_size = run["batch_size"]
val_bs = min(batch_size, max(1, len(Xva)))
val_ds = make_dataset(Xva, batch_size=val_bs, shuffle=False, drop_remainder=False) if len(Xva) > 0 else None
print(f"Built windows in {time.perf_counter() - stage_t0:.1f}s")
print(f"Train windows: {len(Xtr):,}")
print(f"Validation windows used: {len(Xva):,}/{val_total_windows:,}")


## 13. Prepare a visible generation example

The prompt and held-out continuation are prepared before training so the same example can be inspected throughout the run.


In [ ]:
sample_start = min(263, max(0, len(ids_va) - cfg.context_length))
prompt_len = min(24, cfg.context_length)
sample_prompt = tok.decode(ids_va[sample_start:sample_start + prompt_len])
sample_truth_text = tok.decode(ids_va[sample_start + prompt_len:sample_start + cfg.context_length])
print("Prompt:", prettify_text(sample_prompt))
print("Held-out continuation:", prettify_text(sample_truth_text[:400]))


### Forward corruption example

Before training, inspect how the same validation sequence looks at three mask rates. `[M]` denotes the dedicated diffusion `MASK` token.


In [ ]:
# Visualize the forward (masking) process on one real validation window:
# each token is independently replaced by MASK with probability t.
x_demo = tf.constant(Xva[:1])
for t in (0.15, 0.5, 0.9):
    xt_demo, _, _ = mask_with_given_t(x_demo, cfg.mask_token_id, tf.constant([t], tf.float32))
    ids = xt_demo.numpy()[0].tolist()[:48]  # first 48 tokens keep the printout short
    shown = "".join("[M]" if i == cfg.mask_token_id else tok.decode([i]) for i in ids)
    print(f"t={t:.2f}: {shown}\n")


## 14. Train

This is the expensive training stage. It prints optimization progress and announces validation / preview generation explicitly. If execution appears to stall, the last printed stage tells you where it stopped.

The function returns the **best validation EMA checkpoint** when available. Heavy final metrics are intentionally not run here.


In [ ]:
stage_t0 = time.perf_counter()
model, history = train_diffusion(
    cfg,
    val_ds=val_ds,
    rounds=run["rounds"],
    base_learning_rate=run["base_learning_rate"],
    weight_decay=0.0001,
    clipnorm=4.0,
    patience=run["patience"],
    plot_every=1,
    sample_prompt=sample_prompt,
    sample_truth_text=sample_truth_text,
    sample_strategy="low_confidence",
    max_train_steps_per_round=run["max_train_steps_per_round"],
    X_train=Xtr,
    batch_size=batch_size,
    output_dir=run["output_dir"],
    tokenizer=tok,
    progress_every_steps=run["progress_every_steps"],
)
print(f"Training cell finished in {time.perf_counter() - stage_t0:.1f}s")


## 15. Inspect the training result

This cell is cheap. It redraws the history and reports which round supplied the returned checkpoint.


In [ ]:
plot_lm_training_one_panel(
    history["train_loss"], history["val_loss"],
    history["train_ppl"], history["val_ppl"],
    history["lrs"], history["best_round"], logy=False,
)
if history["best_round"] is not None:
    print(f"Best checkpoint round: {history['best_round'] + 1}")
    print(f"Best val CE@t=0.15: {history['best_metric']:.4f}")
if history.get("round_seconds"):
    print(f"Median completed-round time: {np.median(history['round_seconds']):.1f}s")


## 16. Generate from the returned best checkpoint

Generation is separate from training. If this cell is slow, the bottleneck is the reverse-diffusion sampler rather than optimization or validation.


In [ ]:
stage_t0 = time.perf_counter()
print("Generating final sample...", flush=True)
final_txt = llada_generate(
    model, tok, prompt=sample_prompt,
    total_len=cfg.context_length,
    steps=cfg.num_sampling_steps,
    strategy="low_confidence",
    guidance_strength=cfg.guidance_strength,
)
print(f"Generation finished in {time.perf_counter() - stage_t0:.1f}s")
print("\n--- Generated text ---\n", prettify_text(final_txt), sep="")
print("\n--- True text ---\n", prettify_text(sample_prompt + sample_truth_text), sep="")


## 17. Masked bits-per-token evaluation

**BPT** means bits per token. These are fixed-corruption diagnostics and are not autoregressive perplexity. The fully masked `t=1.0` diagnostic is especially useful because generation starts from a fully masked response.


In [ ]:
if val_ds is None:
    print("No validation dataset available.")
else:
    stage_t0 = time.perf_counter()
    print("Evaluating masked BPT...", flush=True)
    bpt_mask50 = evaluate_bpt(model, val_ds, t_eval=0.5, n_mc=run["bpt_n_mc"])
    bpt_mask100 = evaluate_bpt(model, val_ds, t_eval=1.0, n_mc=1)
    print(f"BPT evaluation finished in {time.perf_counter() - stage_t0:.1f}s")
    print(f"masked bpt@t=0.5: {bpt_mask50:.3f}")
    print(f"fully-masked bpt@t=1.0: {bpt_mask100:.3f}")


## 18. Pseudo-perplexity (PPP) — optional expensive diagnostic

PPP masks one position at a time. It is deliberately isolated in its own cell because it can be substantially slower than the other diagnostics. The quick preset evaluates fewer positions per sequence; the full preset uses 128.

You can skip this cell when you only want to verify training and generation.


In [ ]:
if val_ds is None:
    print("No validation dataset available.")
else:
    ppp_positions = run["ppp_positions_per_seq"]
    approx_examples = len(Xva) * min(ppp_positions, cfg.context_length)
    print(f"PPP positions/sequence: {ppp_positions}")
    print(f"Approx. leave-one-out sequence evaluations: {approx_examples:,} (processed in microbatches)")
    stage_t0 = time.perf_counter()
    print("Evaluating PPP...", flush=True)
    bpt_ppp, ppl_ppp = evaluate_ppp(model, val_ds, positions_per_seq=ppp_positions, microbatch=256)
    print(f"PPP finished in {time.perf_counter() - stage_t0:.1f}s")
    print(f"PPP: ppl={ppl_ppp:.1f}, bpt={bpt_ppp:.3f}")
